# ChromaDB local use case

This notebook connects to a local Chroma server (Docker or `chroma run`) and demonstrates adding and querying data.

If you are not running a server, see the embedded alternative near the end.

## Connection settings

This notebook targets the Docker container port mapping (`localhost:8001` by default).
If your server is on a different port, set an env var before running this notebook:

```
export CHROMA_PORT=8001
```

In [3]:
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
print("Heartbeat:", client.heartbeat())

Heartbeat: 1770716820588917334


In [4]:
collection = client.get_or_create_collection("local_demo")

collection.add(
    ids=["doc-1", "doc-2", "doc-3"],
    documents=[
        "Chroma is a vector database for embeddings.",
        "A vector search lets you find similar meaning.",
        "macOS users can run Chroma locally using Docker or chroma run.",
    ],
    metadatas=[
        {"source": "docs", "topic": "chroma"},
        {"source": "docs", "topic": "search"},
        {"source": "docs", "topic": "mac"},
    ],
)

print("Count:", collection.count())

Count: 3


In [5]:
results = collection.query(
    query_texts=["How do I run Chroma locally on a Mac?"],
    n_results=2,
)
results

{'ids': [['doc-3', 'doc-1']],
 'distances': [[0.3023473, 1.2297055]],
 'embeddings': None,
 'metadatas': [[{'topic': 'mac', 'source': 'docs'},
   {'source': 'docs', 'topic': 'chroma'}]],
 'documents': [['macOS users can run Chroma locally using Docker or chroma run.',
   'Chroma is a vector database for embeddings.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [6]:
# Read back specific documents by id
collection.get(ids=["doc-1", "doc-3"])

{'ids': ['doc-1', 'doc-3'],
 'embeddings': None,
 'metadatas': [{'source': 'docs', 'topic': 'chroma'},
  {'source': 'docs', 'topic': 'mac'}],
 'documents': ['Chroma is a vector database for embeddings.',
  'macOS users can run Chroma locally using Docker or chroma run.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

## Optional: Embedded mode (no server)

If you are not running a server, use the in-process client instead:

In [7]:
from chromadb import PersistentClient

embedded_client = PersistentClient(path="./chroma_data")
embedded_collection = embedded_client.get_or_create_collection("local_demo")
print("Embedded count:", embedded_collection.count())

Embedded count: 0


In [8]:
## test 
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))
client = chromadb.HttpClient(host=HOST, port=PORT)
ids = collection.get(ids=["doc-1", "doc-3"])
print(ids)

{'ids': ['doc-1', 'doc-3'], 'embeddings': None, 'metadatas': [{'source': 'docs', 'topic': 'chroma'}, {'source': 'docs', 'topic': 'mac'}], 'documents': ['Chroma is a vector database for embeddings.', 'macOS users can run Chroma locally using Docker or chroma run.'], 'data': None, 'uris': None, 'included': ['metadatas', 'documents']}


In [ ]:
## My Examples from here 

In [19]:
## 1) Collections - 
### Concept: A collection is a named container for vectors + documents + metadata.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.count()

5

In [11]:
## 2) Documents and IDs
### Concept: Each document must have a unique ID.


3

In [13]:
## 3) Metadata
### Concept: Metadata lets you filter search results.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["d4", "d5"],
    documents=[
        "Mac users can run Chroma with Docker.",
        "Server mode uses chroma run.",
    ],
    metadatas=[
        {"topic": "mac", "source": "docs"},
        {"topic": "server", "source": "docs"},
    ],
)

In [14]:
## 4) Similarity Search (Query)
### import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.query(
    query_texts=["How do I run Chroma locally?"],
    n_results=2,
)

{'ids': [['d5', 'd4']],
 'distances': [[0.63203347, 0.64515984]],
 'embeddings': None,
 'metadatas': [[{'source': 'docs', 'topic': 'server'},
   {'topic': 'mac', 'source': 'docs'}]],
 'documents': [['Server mode uses chroma run.',
   'Mac users can run Chroma with Docker.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [16]:
## 5) Metadata Filtering
### Concept: Filter search by metadata fields.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.query(
    query_texts=["How do I run Chroma locally?"],
    n_results=3,
    where={"topic": "mac"},
)

{'ids': [['d4']],
 'distances': [[0.64515984]],
 'embeddings': None,
 'metadatas': [[{'topic': 'mac', 'source': 'docs'}]],
 'documents': [['Mac users can run Chroma with Docker.']],
 'uris': None,
 'data': None,
 'included': ['metadatas', 'documents', 'distances']}

In [17]:
## 6) Get by ID
### Concept: Fetch stored documents directly by ID
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.get(ids=["d1", "d4"])

{'ids': ['d1', 'd4'],
 'embeddings': None,
 'metadatas': [None, {'source': 'docs', 'topic': 'mac'}],
 'documents': ['Chroma is a vector database.',
  'Mac users can run Chroma with Docker.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

In [18]:
## 7) Update Documents
### Concept: Update a document by re-adding with the same ID.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["d2"],
    documents=["Vectors represent meaning and context."],
    metadatas=[{"topic": "vectors"}],
)
collection.get(ids=["d2"])

{'ids': ['d2'],
 'embeddings': None,
 'metadatas': [None],
 'documents': ['Vectors represent meaning.'],
 'data': None,
 'uris': None,
 'included': ['metadatas', 'documents']}

In [21]:
## 8) Delete Documents
### Concept: Remove documents by ID.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.delete(ids=["d3"])
collection.count()    

4

In [22]:
## 9) Collections List and Delete
### Concept: You can list and remove entire collections.
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
client.list_collections()

[Collection(name=concepts_demo), Collection(name=local_demo)]

In [7]:
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
client.delete_collection("concepts_demo")
client.list_collections()

[Collection(name=local_demo), Collection(name=faq_unstructured)]

In [2]:
## 11) Embeddings (What happens behind the scenes)
### Concept: Chroma stores vectors computed from text. If you pass raw text, Chroma uses its default embedding function unless configured otherwise.

### You can also supply your own embeddings (example uses dummy vectors):
import chromadb

client = chromadb.HttpClient(host="localhost", port=8001)
collection = client.get_or_create_collection("concepts_demo")
collection.add(
    ids=["e1", "e2"],
    documents=["alpha", "beta"],
    embeddings=[[0.1, 0.2, 0.3], [0.2, 0.1, 0.0]],
)

In [11]:
## Extract the faqs from pdf and store them into chromadb
import os
import re
import chromadb
from pypdf import PdfReader

PDF_PATH = "/Users/tsm/work/learn-perosnal/vectordb-chroma/faq-data/faq_unstructured.pdf"

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_or_create_collection("faq_unstructured")

# ---- Extract text from PDF ----
reader = PdfReader(PDF_PATH)
raw_pages = [(i, (p.extract_text() or "")) for i, p in enumerate(reader.pages)]
raw_text = "\n".join(text for _, text in raw_pages)

if not raw_text.strip():
    raise ValueError("No text extracted from PDF. Is it scanned? Consider OCR.")

# ---- Clean text ----
# Fix hyphenated line breaks: "instruc-\n tions" -> "instructions"
raw_text = re.sub(r"-\n(\w)", r"\1", raw_text)
# Normalize newlines
raw_text = re.sub(r"\r\n?", "\n", raw_text)

lines = [ln.strip() for ln in raw_text.splitlines() if ln.strip()]

def is_question(line: str) -> bool:
    if not line:
        return False
    if line.lower().startswith("frequently asked questions"):
        return False
    # Most FAQ questions end with '?'
    if line.endswith("?"):
        return True
    # Heuristic fallback
    return bool(re.match(r"^(How|What|Why|When|Where|Who|Can|Is|Do|Does|Are|Will)\b", line))

def clean(s: str) -> str:
    return " ".join(s.split()).strip()

documents, metadatas, ids = [], [], []
current_q = None
current_a_lines = []
row_idx = 0

for ln in lines:
    if is_question(ln):
        # flush previous Q/A
        if current_q and current_a_lines:
            answer = clean(" ".join(current_a_lines))
            documents.append(f"Question: {clean(current_q)}\nAnswer: {answer}")
            metadatas.append({"source": "faq_unstructured.pdf", "row_id": row_idx})
            ids.append(f"faq-pdf-{row_idx:06d}")
            row_idx += 1
        current_q = ln
        current_a_lines = []
    else:
        # accumulate answer lines
        if current_q:
            current_a_lines.append(ln)

# flush last
if current_q and current_a_lines:
    answer = clean(" ".join(current_a_lines))
    documents.append(f"Question: {clean(current_q)}\nAnswer: {answer}")
    metadatas.append({"source": "faq_unstructured.pdf", "row_id": row_idx})
    ids.append(f"faq-pdf-{row_idx:06d}")

if not documents:
    print("Sample extracted lines:")
    for ln in lines[:50]:
        print(ln)
    raise ValueError("No Q/A pairs parsed. Update parsing rules to match PDF format.")

# ---- Upsert into Chroma ----
collection.upsert(ids=ids, documents=documents, metadatas=metadatas)

print("Loaded rows:", len(documents))
print("Collection count:", collection.count())

Loaded rows: 13
Collection count: 13


In [12]:
## Fetch all the extracted faqs from chromadb 
import os
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_collection("faq_unstructured")

# Fetch first 5 FAQs (by id prefix)
result = collection.get(
    include=["documents", "metadatas"],
    limit=5
)
result

{'ids': ['faq-pdf-000000',
  'faq-pdf-000001',
  'faq-pdf-000002',
  'faq-pdf-000003',
  'faq-pdf-000004'],
 'embeddings': None,
 'metadatas': [{'row_id': 0, 'source': 'faq_unstructured.pdf'},
  {'source': 'faq_unstructured.pdf', 'row_id': 1},
  {'row_id': 2, 'source': 'faq_unstructured.pdf'},
  {'source': 'faq_unstructured.pdf', 'row_id': 3},
  {'source': 'faq_unstructured.pdf', 'row_id': 4}],
 'documents': ["Question: How do I reset my password?\nAnswer: Go to the login page, click on 'Forgot Password', and follow the instructions sent to your registered email address.",
  'Question: How can I change my email address?\nAnswer: Navigate to Account Settings, update your email address, and verify it using the confirmation link sent to the new email.',
  'Question: How do I contact customer support?\nAnswer: You can reach customer support via the Help section in the application or by emailing support@example.com.',
  'Question: What is your refund policy?\nAnswer: Refunds are available w

In [13]:
## 1) Semantic FAQ search (top‑k)
import os
import re
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_collection("faq_unstructured")

def parse_qa(doc: str):
    # robust split
    parts = re.split(r"\nAnswer:\s*", doc, maxsplit=1, flags=re.IGNORECASE)
    q = parts[0].replace("Question:", "").strip() if parts else ""
    a = parts[1].strip() if len(parts) > 1 else ""
    return q, a

def faq_search(query: str, k: int = 3):
    res = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        q, a = parse_qa(doc)
        out.append({
            "question": q,
            "answer": a,
            "distance": dist,
            "row_id": meta.get("row_id"),
            "source": meta.get("source"),
        })
    return out

# Example
faq_search("I forgot my password", k=3)

[{'question': 'How do I reset my password?',
  'answer': "Go to the login page, click on 'Forgot Password', and follow the instructions sent to your registered email address.",
  'distance': 0.6449262,
  'row_id': 0,
  'source': 'faq_unstructured.pdf'},
 {'question': 'How do I delete my account?',
  'answer': 'Account deletion can be requested from Account Settings. This action is permanent and cannot be reversed.',
  'distance': 1.3793726,
  'row_id': 8,
  'source': 'faq_unstructured.pdf'},
 {'question': 'How do I contact customer support?',
  'answer': 'You can reach customer support via the Help section in the application or by emailing support@example.com.',
  'distance': 1.3951215,
  'row_id': 2,
  'source': 'faq_unstructured.pdf'}]

In [14]:
## 2) Chatbot‑style reply (best match)
def faq_chat_reply(user_question: str, k: int = 3, max_distance: float = 0.8):
    hits = faq_search(user_question, k=k)
    if not hits:
        return "I don't know based on the FAQ. Can you clarify your question?"

    best = hits[0]
    if best["distance"] > max_distance:
        return "I don't know based on the FAQ. Can you clarify your question?"

    return (
        f"{best['answer']}\n"
        f"(SOURCE: {best['source']}, ROW: {best['row_id']})"
    )

# Example
print(faq_chat_reply("How do I reset my password?"))

Go to the login page, click on 'Forgot Password', and follow the instructions sent to your registered email address.
(SOURCE: faq_unstructured.pdf, ROW: 0)


In [15]:
## runnable multi‑hit FAQ example (connect → search → combined answer)
import os
import re
import chromadb

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_collection("faq_unstructured")

def parse_qa(doc: str):
    parts = re.split(r"\nAnswer:\s*", doc, maxsplit=1, flags=re.IGNORECASE)
    q = parts[0].replace("Question:", "").strip() if parts else ""
    a = parts[1].strip() if len(parts) > 1 else ""
    return q, a

def faq_search(query: str, k: int = 5):
    res = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    out = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        q, a = parse_qa(doc)
        out.append({
            "question": q,
            "answer": a,
            "distance": dist,
            "row_id": meta.get("row_id"),
            "source": meta.get("source"),
        })
    return out

def faq_chat_reply_multi(user_question: str, k: int = 3, max_distance: float = 0.8):
    hits = faq_search(user_question, k=k)
    if not hits:
        return "I don't know based on the FAQ. Can you clarify your question?"

    good = [h for h in hits if h["distance"] <= max_distance]
    if not good:
        return "I don't know based on the FAQ. Can you clarify your question?"

    top = good[:3]
    combined = "\n\n".join(
        f"- {h['answer']} (SOURCE: {h['source']}, ROW: {h['row_id']})"
        for h in top
    )
    return combined

# Example usage
print(faq_chat_reply_multi("How do I reset my password?"))
print(faq_chat_reply_multi("I want to cancel my subscription"))

- Go to the login page, click on 'Forgot Password', and follow the instructions sent to your registered email address. (SOURCE: faq_unstructured.pdf, ROW: 0)
- Open Billing Settings and click on 'Cancel Subscription'. Your access remains active until the current billing cycle ends. (SOURCE: faq_unstructured.pdf, ROW: 4)


In [17]:
## OCR → Llama → Chroma snippet, now reading all Ollama configs from env (URL, model, timeout, retries)
## standalone streaming + chunked OCR snippet
import os
import json
import re
import time
import urllib.request
from pdf2image import convert_from_path
import pytesseract
import chromadb

PDF_PATH = "/Users/tsm/work/learn-perosnal/vectordb-chroma/faq-data/faq_unstructured.pdf"

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:latest")
OLLAMA_TIMEOUT = int(os.getenv("OLLAMA_TIMEOUT", "30"))
OLLAMA_RETRIES = int(os.getenv("OLLAMA_RETRIES", "5"))

HOST = os.getenv("CHROMA_HOST", "localhost")
PORT = int(os.getenv("CHROMA_PORT", "8001"))

client = chromadb.HttpClient(host=HOST, port=PORT)
collection = client.get_or_create_collection("faq_ocr_llama")

# ---- 1) OCR the PDF ----
images = convert_from_path(PDF_PATH, dpi=300)
ocr_text = "\n".join(pytesseract.image_to_string(img) for img in images).strip()
if not ocr_text:
    raise ValueError("OCR produced no text. Check Tesseract install or PDF quality.")

# ---- 2) Chunk OCR text to reduce prompt size ----
def chunk_text(text: str, max_chars: int = 6000):
    parts = []
    buf = []
    size = 0
    for line in text.splitlines():
        if size + len(line) + 1 > max_chars and buf:
            parts.append("\n".join(buf))
            buf = []
            size = 0
        buf.append(line)
        size += len(line) + 1
    if buf:
        parts.append("\n".join(buf))
    return parts

chunks = chunk_text(ocr_text, max_chars=6000)

def ollama_generate_stream(prompt: str) -> str:
    payload = json.dumps({
        "model": OLLAMA_MODEL,
        "prompt": prompt,
        "stream": True
    }).encode("utf-8")
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"}
    )
    out = []
    with urllib.request.urlopen(req, timeout=OLLAMA_TIMEOUT) as resp:
        for line in resp:
            data = json.loads(line.decode("utf-8"))
            if "response" in data:
                out.append(data["response"])
            if data.get("done"):
                break
    return "".join(out).strip()

def extract_qa_pairs(text: str):
    prompt = f"""
You are a strict JSON extractor.

From the text below, extract FAQ question/answer pairs.
Return ONLY valid JSON array of objects like:
[
  {{"question": "...", "answer": "..."}}
]

Rules:
- Keep answers concise but complete.
- Ignore headings or unrelated text.
- If no Q/A pairs, return [].

TEXT:
{text}
"""
    last_err = None
    for attempt in range(OLLAMA_RETRIES):
        try:
            raw = ollama_generate_stream(prompt)
            json_text = raw[raw.find("["): raw.rfind("]") + 1]
            return json.loads(json_text)
        except Exception as e:
            last_err = e
            time.sleep(1)
    raise last_err

all_pairs = []
for chunk in chunks:
    all_pairs.extend(extract_qa_pairs(chunk))

documents, metadatas, ids = [], [], []
for idx, item in enumerate(all_pairs):
    q = re.sub(r"\s+", " ", item.get("question", "")).strip()
    a = re.sub(r"\s+", " ", item.get("answer", "")).strip()
    if not q or not a:
        continue
    documents.append(f"Question: {q}\nAnswer: {a}")
    metadatas.append({"source": "faq_unstructured.pdf", "row_id": idx})
    ids.append(f"faq-ocr-{idx:06d}")

collection.upsert(ids=ids, documents=documents, metadatas=metadatas)
print("Loaded rows:", len(documents))
print("Collection count:", collection.count())

Loaded rows: 13
Collection count: 13
